In [2]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import sys
import glob
from PIL import Image

def validate_image_paths(wolf_snow_paths, husky_no_snow_paths):
    """
    Validate that image paths exist and are non-empty
    
    Args:
        wolf_snow_paths (list): Paths to wolf images with snow
        husky_no_snow_paths (list): Paths to husky images without snow
    
    Raises:
        ValueError: If no images are found or paths are invalid
    """
    if not wolf_snow_paths:
        raise ValueError("""
NO WOLF IMAGES FOUND! 

You need to create a directory structure like:
./wolf_husky_data/
    train/
        wolves_snow/
            # Place wolf images with snow backgrounds here
        huskies_no_snow/
            # Place husky images without snow backgrounds here

Detailed steps:
1. Create the directory structure
2. Add wolf images with snow backgrounds to wolves_snow/
3. Add husky images without snow to huskies_no_snow/
4. Ensure images are .jpg or .png
5. Make sure the paths are correct
""")
    
    if not husky_no_snow_paths:
        raise ValueError("""
NO HUSKY IMAGES FOUND!

Please add husky images to the huskies_no_snow/ directory
""")
    
    # Additional checks for image validity
    valid_wolf_paths = [p for p in wolf_snow_paths if p.is_file()]
    valid_husky_paths = [p for p in husky_no_snow_paths if p.is_file()]
    
    if not valid_wolf_paths:
        raise ValueError("No valid wolf image files found. Check file formats and paths.")
    
    if not valid_husky_paths:
        raise ValueError("No valid husky image files found. Check file formats and paths.")



2025-03-04 14:53:57.834740: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-04 14:53:57.841178: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-04 14:53:57.859211: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741121637.892153   46112 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741121637.901527   46112 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-04 14:53:57.934431: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

In [3]:
import os

In [4]:
def extract_features(preprocessed_images):
    
    # Load Inception model (without top layers)
    base_model = InceptionV3(weights='imagenet', include_top=False)
    
    # Find the first max pooling layer
    max_pooling_layers = [layer for layer in base_model.layers if 'max_pooling2d' in layer.name]
    
    if not max_pooling_layers:
        raise ValueError("No max pooling layers found in the model")
    
    # Get the first max pooling layer
    first_max_pooling_layer = max_pooling_layers[0]
    
    # Create feature extractor
    feature_extractor = tf.keras.Model(
        inputs=base_model.input,
        outputs=first_max_pooling_layer.output
    )
    
    # Extract features
    features = feature_extractor.predict(preprocessed_images)
    
    # Flatten the features for use with logistic regression
    flattened_features = features.reshape(features.shape[0], -1)
    return flattened_features


In [5]:
def load_and_preprocess_images(image_paths, target_size=(299, 299)):
    images = []
    for img_path in image_paths:
        img = Image.open(img_path).convert('RGB')
        img = img.resize(target_size)
        img_array = np.array(img)
        images.append(img_array)
    
    images = np.array(images)
    # Preprocess for Inception model
    preprocessed_images = preprocess_input(images)
    return preprocessed_images

In [6]:



def train_biased_classifier(wolf_snow_paths, husky_no_snow_paths):
    # Combine image paths and create labels
    image_paths = wolf_snow_paths + husky_no_snow_paths
    labels = np.array([1] * len(wolf_snow_paths) + [0] * len(husky_no_snow_paths))
    
    # Load and preprocess images
    preprocessed_images = load_and_preprocess_images(image_paths)
    
    # Extract features
    features = extract_features(preprocessed_images)
    
    # Train logistic regression
    model = LogisticRegression(random_state=42)
    model.fit(features, labels)
    
    return model, features

In [7]:
# Function to evaluate the model
def evaluate_model(model, image_paths, true_labels):
    preprocessed_images = load_and_preprocess_images(image_paths)
    features = extract_features(preprocessed_images)
    
    predictions = model.predict(features)
    accuracy = accuracy_score(true_labels, predictions)
    report = classification_report(true_labels, predictions, target_names=['Husky', 'Wolf'])
    
    return predictions, accuracy, report

In [11]:
def visualize_with_lime(model, image_paths, predictions, true_labels):
    try:
        import lime
        from lime import lime_image
        from skimage.segmentation import mark_boundaries
        
        explainer = lime_image.LimeImageExplainer(
                        kernel_width=0.5,  # Adjust kernel width
                        verbose=False,     # Reduce verbosity
        )
        
        for i, img_path in enumerate(image_paths):
            img = Image.open(img_path).convert('RGB')
            img = img.resize((299, 299))
            img_array = np.array(img)
            
            # Define prediction function for LIME
            def predict_fn(images):
                preprocessed = preprocess_input(images)
                features = extract_features(preprocessed)
                return model.predict_proba(features)
            
            explanation = explainer.explain_instance(
                img_array, 
                predict_fn,
                top_labels=2, 
                hide_color=0, 
                num_samples=100
            )
            
            # Get the explanation for the predicted class
            pred_class = int(predictions[i])
            
            # Plot the explanation
            plt.figure(figsize=(10, 5))
            
            plt.subplot(1, 2, 1)
            plt.imshow(img_array)
            class_name = "Wolf" if pred_class == 1 else "Husky"
            true_class = "Wolf" if true_labels[i] == 1 else "Husky"
            plt.title(f"Prediction: {class_name}, True: {true_class}")
            
            plt.subplot(1, 2, 2)
            temp, mask = explanation.get_image_and_mask(
                pred_class, 
                positive_only=True, 
                num_features=5, 
                hide_rest=False
            )
            plt.imshow(mark_boundaries(temp / 255.0, mask))
            plt.title("LIME explanation")
            
            plt.tight_layout()
            plt.show()
            
    except ImportError:
        print("LIME package not installed. Run 'pip install lime' to use this function.")



In [13]:


def run_wolf_husky_experiment(data_dir):
    """
    Expected directory structure:
    data_dir/
        train/
            wolves_snow/     # Wolves in snow
            huskies_no_snow/ # Huskies without snow
    """
    # Convert to Path object
    train_dir_w = f'/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/train/wolves_snow/'
    train_dir_h = f'/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/train/huskies_no_snow/'
    data_path = Path(data_dir)
    
    # Load training data (biased dataset)
    wolf_snow_paths = [f'{train_dir_w}im{i}.jpg' for i in range(1,11)]
    wolf_snow_paths = list(map(Path, wolf_snow_paths))
    
    husky_no_snow_paths = [f'{train_dir_h}husky{i}.jpg' for i in range(1,12)]
    husky_no_snow_paths = list(map(Path, husky_no_snow_paths))
    
    #wolf_snow_paths = list(map(str, data_path.glob('train/wolves_snow/*.[jJ][pP][gG]*')))
    #husky_no_snow_paths = list(map(str, data_path.glob('train/huskies_no_snow/*.[jJ][pP]*')))
    
    # Validate image paths before proceeding
    validate_image_paths(wolf_snow_paths, husky_no_snow_paths)
    
    print(f"Training with {len(wolf_snow_paths)} wolves (with snow) and {len(husky_no_snow_paths)} huskies (without snow)")
    
    # Train biased model
    model, train_features = train_biased_classifier(wolf_snow_paths, husky_no_snow_paths)
    
    
    
    ####################################################################################
    # Prepare test set
    test_dir_ws = f'/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/wolves_snow/'
    test_dir_wns = f'/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/wolves_no_snow/'
    test_dir_hs = f'/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/huskies_snow/'
    test_dir_hns = f'/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/huskies_no_snow/'
    wolf_snow_test_paths = [f'{test_dir_ws}wolf{i}.jpg' for i in range(1,6)]
    wolf_snow_test_paths = list(map(Path, wolf_snow_test_paths))
    
    wolf_no_snow_test_paths = [f'{test_dir_wns}wolf.jpg' for i in range(1,2)]
    wolf_no_snow_test_paths = list(map(Path, wolf_no_snow_test_paths))
    
    husky_snow_test_paths = [f'{test_dir_hs}husky.jpg' for i in range(1,2)]
    husky_snow_test_paths = list(map(Path, husky_snow_test_paths))
    
    
    husky_no_snow_test_paths = [f'{test_dir_hns}husky{i}.jpg' for i in range(1,5)]
    husky_no_snow_test_paths = list(map(Path, husky_no_snow_test_paths))
    # Create balanced test set as described in the paper
    balanced_test_paths = (wolf_snow_test_paths[:4] + wolf_no_snow_test_paths[:1] + 
                          husky_snow_test_paths[:1] + husky_no_snow_test_paths[:4])
    
    # Create true labels for balanced test set
    # 1 for wolf, 0 for husky
    balanced_test_labels = np.array([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
    
    # Evaluate on balanced test set
    predictions, accuracy, report = evaluate_model(model, balanced_test_paths, balanced_test_labels)
    print(f'prediction is {predictions}')
    # Print results
    print(f"Test Accuracy: {accuracy:.2f}")
    print("Classification Report:")
    print(report)
    
    # Print model coefficients
    print("Model Coefficients:")
    print(f"Intercept: {model.intercept_}")
    print(f"Top 5 positive weights: {model.coef_[0].argsort()[-5:]}")
    print(f"Top 5 negative weights: {model.coef_[0].argsort()[:5]}")
    
    # Show model predictions without explanations
    print("\nTest Predictions without Explanations:")
    for i, path in enumerate(balanced_test_paths):
        img_name = os.path.basename(path)
        pred_class = "Wolf" if predictions[i] == 1 else "Husky"
        true_class = "Wolf" if balanced_test_labels[i] == 1 else "Husky"
        print(f"Image: {img_name}, Prediction: {pred_class}, True: {true_class}")
    
    # Visualize with LIME
    print("\nGenerating LIME explanations...")
    #visualize_with_lime(model, balanced_test_paths, predictions, balanced_test_labels)
    
    return model, balanced_test_paths, predictions, balanced_test_labels



In [14]:
data_dir = "./data_dir"
run_wolf_husky_experiment(data_dir)

Training with 10 wolves (with snow) and 11 huskies (without snow)
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 452ms/step
prediction is [1 1 1 1 0 1 0 1 0 0]
Test Accuracy: 0.70
Classification Report:
              precision    recall  f1-score   support

       Husky       0.75      0.60      0.67         5
        Wolf       0.67      0.80      0.73         5

    accuracy                           0.70        10
   macro avg       0.71      0.70      0.70        10
weighted avg       0.71      0.70      0.70        10

Model Coefficients:
Intercept: [2.15529952e-05]
Top 5 positive weights: [284811 137481 275341 284821 112129]
Top 5 negative weights: [312223 312287 307551 307862 307743]

Test Predictions without Explanations:
Image: wolf1.jpg, Prediction: Wolf, True: Wolf
Image: wolf2.jpg, Prediction: Wolf, True: Wolf
Image: wolf3.jpg, Prediction: Wolf, True: Wolf
Image: wolf4.jpg, Prediction: Wolf, True: Wolf
Image: wolf.jpg, Prediction: Husky, True: Wolf
Image

(LogisticRegression(random_state=42),
 [PosixPath('/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/wolves_snow/wolf1.jpg'),
  PosixPath('/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/wolves_snow/wolf2.jpg'),
  PosixPath('/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/wolves_snow/wolf3.jpg'),
  PosixPath('/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/wolves_snow/wolf4.jpg'),
  PosixPath('/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/wolves_no_snow/wolf.jpg'),
  PosixPath('/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/huskies_snow/husky.jpg'),
  PosixPath('/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/test/huskies_no_snow/husky1.jpg'),
  PosixPath('/home/bipar001/anaconda3/python_test/huskyvswolf/ModelAugmentation/src/data_dir/